
# 🛒 LangChain + FastMCP + SQLite + OpenAI en local
## Servidor MCP de Analítica E-commerce con tools SQL explícitas

**Material de clase — MCP Clase 2**

Este notebook está diseñado para que sea visible la relación entre:

```text
Pregunta de negocio
      ↓
Tool MCP
      ↓
Consulta SQL concreta y parametrizada
      ↓
SQLite
      ↓
Resultado para el LLM
      ↓
Respuesta final del agente LangChain
```

La idea pedagógica no es dar al modelo una herramienta genérica como `ejecutar_sql(sql)`.  
Cada tool representa una **capacidad de negocio explícita**, con una consulta SQL identificable y controlada.

```text
Usuario
  ↓
Agente LangChain + ChatOpenAI
  ↓
Tools adaptadas desde MCP
  ↓
FastMCP Server por HTTP
  ↓
SQLite: tabla orders
```

> **Por qué HTTP y no STDIO:** algunos entornos Jupyter pueden fallar con `UnsupportedOperation: fileno` al usar STDIO. HTTP evita ese problema y conserva exactamente el mismo protocolo MCP.



---
## Paso 1 — Instalar dependencias


In [ ]:

# En Colab se usaba:
# !pip -q install -U fastmcp langchain langchain-openai langchain-mcp-adapters pandas

# En local/Jupyter usa %pip para instalar en el kernel activo.
%pip install -U fastmcp langchain langchain-openai langchain-mcp-adapters pandas

print("✅ Dependencias instaladas")



---
## Paso 2 — Configurar OpenAI y cargar el dataset local

Esta versión local usa el CSV que ya está en el mismo folder del notebook:
`ecommerceordersdataset(23062026152737).csv`.


In [ ]:

import os
from getpass import getpass
from pathlib import Path

# En Colab se usaba:
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = next(name for name in uploaded if name.lower().endswith(".csv"))

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

# Puedes reemplazar este valor por un modelo habilitado en tu cuenta.
os.environ.setdefault("OPENAI_MODEL", "gpt-4.1-mini")

BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "ecommerceordersdataset(23062026152737).csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encontró el CSV esperado: {CSV_PATH}")

print("✅ Dataset cargado:", CSV_PATH)
print("✅ Modelo:", os.environ["OPENAI_MODEL"])



---
## Paso 3 — Crear SQLite y revisar la tabla

Primero llevamos el CSV a una tabla SQLite llamada `orders`.  
Esta tabla será la única fuente de datos del MCP Server.


In [ ]:

import sqlite3
import pandas as pd
from pathlib import Path

# En Colab se usaba:
# DB_PATH = Path("/content/ecommerce_demo.db").resolve()
DB_PATH = (BASE_DIR / "ecommerce_demo.db").resolve()

df = pd.read_csv(CSV_PATH)
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce").dt.strftime("%Y-%m-%d")

with sqlite3.connect(DB_PATH) as conn:
    df.to_sql("orders", conn, if_exists="replace", index=False)

    # Índices: hacen más rápidas las consultas que usarán las tools.
    conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(Customer_ID)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_country ON orders(Country)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_segment ON orders(Customer_Segment)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_date ON orders(Order_Date)")

print(f"✅ Base SQLite creada: {DB_PATH}")
print(f"✅ Filas cargadas: {len(df):,}")
print("\nColumnas disponibles:")
print(list(df.columns))

display(df.head(3))



---
## Paso 4 — Mapa pedagógico: pregunta → tool → SQL

| Pregunta del usuario | Tool MCP | Qué consulta SQL realiza |
|---|---|---|
| “Busca clientes Premium” | `buscar_clientes` | Filtra `Customer_ID`, país, ciudad o segmento |
| “¿Cuánto consume este cliente?” | `resumen_consumo_cliente` | Agrega órdenes, gasto, unidades, descuentos y fechas |
| “¿Qué ha comprado últimamente?” | `ordenes_recientes_cliente` | Lista órdenes individuales ordenadas por fecha |
| “¿Qué categorías prefiere?” | `consumo_por_categoria` | Agrupa gasto y unidades por categoría |
| “¿Tiene problemas de servicio?” | `metricas_experiencia_cliente` | Calcula devoluciones, rating y entrega |
| “¿Qué países venden más?” | `ranking_ventas_por_pais` | Agrupa facturación y utilidad por país |

A continuación, cada función tendrá dentro de sí su propia consulta SQL.  
Así se vuelve evidente qué capacidad entrega cada tool y qué parte de la base utiliza.



---
## Paso 5 — Construir el FastMCP Server

Lee cuidadosamente cada bloque:

1. El decorador `@mcp.tool()` convierte la función Python en una tool MCP.
2. El docstring se vuelve parte de la descripción que verá LangChain y el LLM.
3. Los argumentos tipados se convierten en el schema de entrada de la tool.
4. Cada tool contiene una consulta SQL concreta, parametrizada y visible.
5. No existe acceso a SQL libre desde el LLM.


In [ ]:

from pathlib import Path

# En Colab se usaba:
# %%writefile /content/ecommerce_mcp_server.py

SERVER_PATH = (BASE_DIR / "ecommerce_mcp_server.py").resolve()
SERVER_CODE = '"""\nServidor MCP de Analítica E-commerce\n------------------------------------\nCada tool corresponde a una pregunta de negocio y a una consulta SQL explícita.\n\nTransporte: HTTP / Streamable HTTP\nEndpoint esperado: http://127.0.0.1:8000/mcp\n"""\n\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom fastmcp import FastMCP\n\nDB_PATH = Path(__file__).resolve().with_name("ecommerce_demo.db")\n\nmcp = FastMCP(\n    name="Ecommerce Analytics MCP",\n    instructions=(\n        "Servidor de análisis de e-commerce. "\n        "Todas las herramientas son de solo lectura. "\n        "Usa buscar_clientes si no conoces el Customer_ID exacto."\n    ),\n)\n\n\ndef ejecutar_sql(sql: str, parametros: tuple = ()) -> list[dict]:\n    """\n    Helper técnico: abre SQLite y transforma las filas a diccionarios.\n    La consulta SQL NO llega desde el LLM: cada tool define su propio SQL.\n    """\n    with sqlite3.connect(DB_PATH) as conn:\n        conn.row_factory = sqlite3.Row\n        filas = conn.execute(sql, parametros).fetchall()\n    return [dict(fila) for fila in filas]\n\n\n@mcp.tool()\ndef buscar_clientes(texto: str, limite: int = 10) -> str:\n    """\n    Busca clientes por Customer_ID, país, ciudad o segmento.\n\n    Pregunta que resuelve:\n    - "Busca clientes Premium"\n    - "Encuentra clientes de Chile"\n    - "¿Existe el cliente CUST007322?"\n\n    Args:\n        texto: Texto de búsqueda, por ejemplo "Premium", "Chile" o "CUST007322".\n        limite: Máximo de clientes a devolver. Entre 1 y 25.\n    """\n    limite = max(1, min(limite, 25))\n    patron = f"%{texto.strip()}%"\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Customer_ID,\n            MAX(Country) AS Country,\n            MAX(City) AS City,\n            MAX(Customer_Segment) AS Customer_Segment,\n            MAX(Membership_Status) AS Membership_Status,\n            COUNT(*) AS Total_Orders,\n            ROUND(SUM(Order_Amount), 2) AS Total_Spent\n        FROM orders\n        WHERE Customer_ID LIKE ?\n           OR Country LIKE ?\n           OR City LIKE ?\n           OR Customer_Segment LIKE ?\n        GROUP BY Customer_ID\n        ORDER BY Total_Spent DESC\n        LIMIT ?\n    """\n\n    filas = ejecutar_sql(sql, (patron, patron, patron, patron, limite))\n    return json.dumps(filas or [{"message": "No se encontraron clientes"}], ensure_ascii=False)\n\n\n@mcp.tool()\ndef resumen_consumo_cliente(customer_id: str) -> str:\n    """\n    Resume el consumo histórico de un cliente identificado por Customer_ID exacto.\n\n    Pregunta que resuelve:\n    - "¿Cuánto ha consumido CUST007322?"\n    - "¿Cuál es su ticket promedio?"\n    - "¿Desde cuándo compra este cliente?"\n\n    Args:\n        customer_id: Identificador exacto del cliente, por ejemplo "CUST007322".\n    """\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Customer_ID,\n            COUNT(*) AS Total_Orders,\n            ROUND(SUM(Order_Amount), 2) AS Total_Spent,\n            ROUND(AVG(Order_Amount), 2) AS Average_Order_Value,\n            SUM(Quantity) AS Units_Purchased,\n            ROUND(SUM(Discount_Amount), 2) AS Total_Discounts,\n            MIN(Order_Date) AS First_Order_Date,\n            MAX(Order_Date) AS Last_Order_Date,\n            ROUND(MAX(Customer_Lifetime_Value), 2) AS Customer_Lifetime_Value\n        FROM orders\n        WHERE Customer_ID = ?\n        GROUP BY Customer_ID\n    """\n\n    filas = ejecutar_sql(sql, (customer_id,))\n    return json.dumps(filas or [{"message": "Cliente no encontrado"}], ensure_ascii=False)\n\n\n@mcp.tool()\ndef ordenes_recientes_cliente(customer_id: str, limite: int = 5) -> str:\n    """\n    Lista las órdenes más recientes de un cliente.\n\n    Pregunta que resuelve:\n    - "¿Qué compró últimamente CUST007322?"\n    - "¿Cuál fue su último pedido?"\n    - "¿Qué productos adquirió y cuánto pagó?"\n\n    Args:\n        customer_id: Identificador exacto del cliente.\n        limite: Máximo de órdenes a devolver. Entre 1 y 20.\n    """\n    limite = max(1, min(limite, 20))\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Order_ID,\n            Order_Date,\n            Product_Category,\n            Product_Subcategory,\n            Brand,\n            Quantity,\n            Unit_Price,\n            Discount_Amount,\n            Order_Amount,\n            Payment_Method,\n            Shipping_Method,\n            Order_Status,\n            Returned\n        FROM orders\n        WHERE Customer_ID = ?\n        ORDER BY Order_Date DESC, Order_ID DESC\n        LIMIT ?\n    """\n\n    filas = ejecutar_sql(sql, (customer_id, limite))\n    return json.dumps(filas or [{"message": "No hay órdenes para este cliente"}], ensure_ascii=False)\n\n\n@mcp.tool()\ndef consumo_por_categoria(customer_id: str) -> str:\n    """\n    Resume cuánto ha gastado un cliente por categoría de producto.\n\n    Pregunta que resuelve:\n    - "¿Qué categorías prefiere CUST007322?"\n    - "¿En qué tipo de productos gasta más?"\n    - "¿Cuáles son sus hábitos de compra?"\n\n    Args:\n        customer_id: Identificador exacto del cliente.\n    """\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Product_Category,\n            COUNT(*) AS Total_Orders,\n            SUM(Quantity) AS Units_Purchased,\n            ROUND(SUM(Order_Amount), 2) AS Total_Spent,\n            ROUND(AVG(Order_Amount), 2) AS Average_Order_Value\n        FROM orders\n        WHERE Customer_ID = ?\n        GROUP BY Product_Category\n        ORDER BY Total_Spent DESC\n    """\n\n    filas = ejecutar_sql(sql, (customer_id,))\n    return json.dumps(filas or [{"message": "Cliente no encontrado"}], ensure_ascii=False)\n\n\n@mcp.tool()\ndef metricas_experiencia_cliente(customer_id: str) -> str:\n    """\n    Calcula métricas de experiencia y postventa de un cliente.\n\n    Pregunta que resuelve:\n    - "¿Tiene muchas devoluciones?"\n    - "¿Cómo evalúa sus compras?"\n    - "¿Ha tenido problemas de despacho?"\n\n    Args:\n        customer_id: Identificador exacto del cliente.\n    """\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Customer_ID,\n            COUNT(*) AS Total_Orders,\n            SUM(CASE WHEN Returned = \'Yes\' THEN 1 ELSE 0 END) AS Returned_Orders,\n            ROUND(\n                100.0 * SUM(CASE WHEN Returned = \'Yes\' THEN 1 ELSE 0 END) / COUNT(*),\n                2\n            ) AS Return_Rate_Percent,\n            ROUND(AVG(Review_Rating), 2) AS Average_Review_Rating,\n            ROUND(AVG(Delivery_Days), 2) AS Average_Delivery_Days,\n            SUM(CASE WHEN Order_Status <> \'Delivered\' THEN 1 ELSE 0 END) AS Non_Delivered_Orders\n        FROM orders\n        WHERE Customer_ID = ?\n        GROUP BY Customer_ID\n    """\n\n    filas = ejecutar_sql(sql, (customer_id,))\n    return json.dumps(filas or [{"message": "Cliente no encontrado"}], ensure_ascii=False)\n\n\n@mcp.tool()\ndef ranking_ventas_por_pais(year: int = 2023, limite: int = 10) -> str:\n    """\n    Entrega un ranking de ventas y utilidad por país para un año.\n\n    Pregunta que resuelve:\n    - "¿Qué países venden más en 2023?"\n    - "¿Dónde se concentra la facturación?"\n    - "¿Qué país genera más utilidad?"\n\n    Args:\n        year: Año a analizar, por ejemplo 2023.\n        limite: Número máximo de países del ranking. Entre 1 y 20.\n    """\n    limite = max(1, min(limite, 20))\n\n    # CONSULTA SQL ASOCIADA A ESTA TOOL:\n    sql = """\n        SELECT\n            Country,\n            COUNT(*) AS Total_Orders,\n            ROUND(SUM(Order_Amount), 2) AS Revenue,\n            ROUND(SUM(Profit_Amount), 2) AS Profit,\n            ROUND(AVG(Order_Amount), 2) AS Average_Order_Value\n        FROM orders\n        WHERE Year = ?\n        GROUP BY Country\n        ORDER BY Revenue DESC\n        LIMIT ?\n    """\n\n    filas = ejecutar_sql(sql, (year, limite))\n    return json.dumps(filas, ensure_ascii=False)\n\n\nif __name__ == "__main__":\n    # HTTP / Streamable HTTP: compatible con Jupyter local.\n    mcp.run(transport="http", host="127.0.0.1", port=8000)\n\n'
SERVER_PATH.write_text(SERVER_CODE, encoding="utf-8")

print(f"✅ Servidor MCP escrito en: {SERVER_PATH}")



---
## Paso 6 — Iniciar el servidor MCP en segundo plano

El servidor se ejecuta como un microservicio local y queda disponible en:

```text
http://127.0.0.1:8000/mcp
```

En una arquitectura de producción, esta misma pieza podría vivir en un contenedor o un servicio cloud.


In [ ]:

import os
import sys
import time
import socket
import subprocess
from pathlib import Path

MCP_HOST = "127.0.0.1"
MCP_PORT = 8000
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
SERVER_PATH = BASE_DIR / "ecommerce_mcp_server.py"
LOG_PATH = BASE_DIR / "ecommerce_mcp_server.log"


def puerto_abierto(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


# En Colab/Linux se usaba:
# !fuser -k 8000/tcp 2>/dev/null || true

if puerto_abierto(MCP_HOST, MCP_PORT):
    raise RuntimeError(
        f"El puerto {MCP_PORT} ya está en uso. Detén el proceso anterior "
        "o cambia MCP_PORT antes de ejecutar esta celda."
    )

log_file = open(LOG_PATH, "w", encoding="utf-8")
server_process = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd=str(BASE_DIR),
)

for _ in range(40):
    time.sleep(0.5)
    if puerto_abierto(MCP_HOST, MCP_PORT):
        break
else:
    print(Path(LOG_PATH).read_text(encoding="utf-8"))
    raise RuntimeError("El servidor FastMCP no logró iniciar.")

print("✅ Servidor MCP activo en:", MCP_URL)
print("✅ PID:", server_process.pid)
print("✅ Log:", LOG_PATH)



---
## Paso 7 — Verificar que MCP publica las tools

Antes de crear el agente, LangChain se conecta al servidor y descubre las tools automáticamente.  
Observa los nombres, descripciones y argumentos: esas definiciones provienen directamente de los decoradores y docstrings del servidor.


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import json

mcp_client = MultiServerMCPClient(
    {
        "ecommerce": {
            "transport": "http",
            "url": MCP_URL,
        }
    }
)

tools = await mcp_client.get_tools()

print(f"📦 Tools MCP descubiertas por LangChain: {len(tools)}\n")

for tool in tools:
    print(f"🔧 {tool.name}")
    print(f"   Descripción: {tool.description}")

    # En esta versión, args_schema ya viene como diccionario JSON Schema.
    schema = tool.args_schema

    # Compatibilidad por si alguna versión entrega un modelo Pydantic.
    if hasattr(schema, "model_json_schema"):
        schema = schema.model_json_schema()

    print("   Schema:")
    print(json.dumps(schema, ensure_ascii=False, indent=2))
    print()


---
## Paso 8 — Probar las tools directamente, antes del LLM

Esta celda es muy importante para la clase: permite mostrar que una tool MCP funciona por sí sola.  
Primero se prueba la capacidad; después se la entrega al agente.

```text
Tool MCP → SQL concreto → SQLite → JSON
```


In [ ]:

tools_por_nombre = {tool.name: tool for tool in tools}

print("─" * 70)
print("🧪 TOOL 1: buscar_clientes('Premium')")
print("─" * 70)
resultado = await tools_por_nombre["buscar_clientes"].ainvoke({"texto": "Premium", "limite": 3})
print(resultado)

print("\n" + "─" * 70)
print("🧪 TOOL 2: resumen_consumo_cliente('CUST007322')")
print("─" * 70)
resultado = await tools_por_nombre["resumen_consumo_cliente"].ainvoke(
    {"customer_id": "CUST007322"}
)
print(resultado)

print("\n" + "─" * 70)
print("🧪 TOOL 3: ranking_ventas_por_pais(2023)")
print("─" * 70)
resultado = await tools_por_nombre["ranking_ventas_por_pais"].ainvoke(
    {"year": 2023, "limite": 5}
)
print(resultado)



---
## Paso 9 — Crear el agente LangChain con OpenAI

LangChain recibe las tools que vienen desde MCP y crea el loop de orquestación:

```text
Pregunta
  ↓
ChatOpenAI selecciona una tool
  ↓
LangChain ejecuta la tool MCP
  ↓
FastMCP ejecuta la query SQL asociada
  ↓
El resultado vuelve al LLM
  ↓
Respuesta final
```

El system prompt funciona como política de uso de herramientas.


In [ ]:

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = """
Eres un analista de e-commerce y debes responder en español.

REGLAS DE ORQUESTACIÓN:
1. Para toda pregunta que requiera datos reales, usa una o más tools MCP antes de responder.
2. Nunca inventes cifras, clientes, productos ni periodos.
3. Si no tienes un Customer_ID exacto, comienza con buscar_clientes.
4. Si la búsqueda devuelve varios clientes y no puedes elegir con certeza, explica la ambigüedad.
5. Para consumo general utiliza resumen_consumo_cliente.
6. Para detalle de órdenes utiliza ordenes_recientes_cliente.
7. Para preferencias de productos utiliza consumo_por_categoria.
8. Para calidad de servicio utiliza metricas_experiencia_cliente.
9. Para ranking de negocio por país utiliza ranking_ventas_por_pais.
10. Todas las herramientas son de solo lectura. Nunca afirmes haber modificado la base.
11. Estructura la respuesta final en:
    - Hallazgos
    - Evidencia
    - Recomendación
"""

llm = ChatOpenAI(
    model=os.environ["OPENAI_MODEL"],
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agente LangChain creado con tools provenientes de MCP.")



---
## Paso 10 — Ejecutar consultas y observar la orquestación

La función imprime qué tools eligió el modelo, con qué argumentos y qué resultado devolvió MCP.


In [ ]:

from langchain.messages import AIMessage, ToolMessage

async def consultar(pregunta: str):
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    print("=" * 80)

    respuesta = await agent.ainvoke(
        {"messages": [{"role": "user", "content": pregunta}]}
    )

    print("\n🔎 TRAZA DE ORQUESTACIÓN")
    for mensaje in respuesta["messages"]:
        if isinstance(mensaje, AIMessage) and mensaje.tool_calls:
            for llamada in mensaje.tool_calls:
                print(f"\n🧠 LLM eligió tool: {llamada['name']}")
                print(f"   Argumentos: {llamada['args']}")
        elif isinstance(mensaje, ToolMessage):
            vista = str(mensaje.content)
            print(f"⚡ MCP devolvió: {vista[:600]}{'...' if len(vista) > 600 else ''}")

    print("\n📝 RESPUESTA FINAL DEL AGENTE")
    print(respuesta["messages"][-1].content)

    return respuesta



### Consulta 1 — El agente debe encadenar varias tools

Primero buscará clientes Premium. Después puede consultar consumo, categorías y experiencia del cliente elegido.


In [ ]:

respuesta_1 = await consultar(
    "Busca clientes Premium. Elige el que tenga mayor gasto total y explícame "
    "su consumo, sus categorías preferidas y si presenta señales de mala experiencia."
)



### Consulta 2 — Análisis agregado de negocio


In [ ]:

respuesta_2 = await consultar(
    "¿Qué cinco países generaron más ventas en 2023? "
    "Indica también cuál parece más atractivo considerando utilidad."
)



### 🎯 Tu turno — Preguntas de prueba

Prueba una por una:

```python
await consultar("¿Cuánto ha consumido CUST007322 y desde cuándo compra?")
await consultar("¿Qué compró últimamente CUST007322?")
await consultar("¿Cuáles son las categorías preferidas de CUST007322?")
await consultar("¿CUST007322 tiene señales de mala experiencia?")
await consultar("Busca clientes de Chile y analiza al que tenga mayor consumo.")
```



---
## Paso 11 — Qué deben retener los alumnos

### 1. Una tool no es “una query cualquiera”
Una tool es una capacidad de negocio con nombre, descripción, argumentos y resultado definidos.

### 2. La query queda encapsulada en el MCP Server
El LLM no ve ni redacta SQL. El desarrollador controla qué datos puede consultar y cómo.

### 3. LangChain orquesta, MCP conecta
- **FastMCP:** publica capacidades.
- **LangChain:** entrega esas capacidades al agente y gestiona el loop.
- **OpenAI:** interpreta la pregunta y decide qué tool usar.
- **SQLite:** guarda y consulta los datos.

### 4. El diseño de herramientas afecta la calidad del agente
Tools con nombres claros, descripciones precisas y una responsabilidad única son más fáciles de seleccionar correctamente.



---
## Limpieza — Detener el servidor al terminar


In [ ]:

if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=10)
    print("🔌 Servidor FastMCP detenido.")
else:
    print("No hay un proceso activo que detener.")


NOTAS: 
Se hizo un website para dar mayor explicación a esta actividad y tienen presnetaciones embebdias para mayor información, dicho website fue desplegado en netlify, se deja url: https://ecommercemcpapp.netlify.app/

Versión colab:
https://drive.google.com/file/d/1hyeNlAnIOgnREU8s1oEb71jqeI0-82FG/view?usp=sharing
